# W03 tutorial — the Data-Quality Report Card

**ACFI20009 Financial Data — 26/27**  
Dr Zeming Li, University of Bristol

**This week you build the Data-Quality Report Card:** a defensible audit of any set of PERMNOs in `crsp_monthly.parquet` — the units trap, group means validated by a second route, an industry join, winsorisation done properly, and the survivorship graveyard quantified — ending in a one-page verdict you can stand behind.

Work straight down the notebook. Each section explains what we are doing and why, then does it. By the last cell you have the whole audit, and the numbers to state it with.

> ### 🤖 Working with Gemini in this notebook
>
> Colab has Gemini built in. You are **encouraged** to use it — reading and
> directing AI-written code is part of what this unit teaches.
>
> **One hard rule:** never paste rows of CRSP data into Gemini (or any other
> AI tool). The licence forbids it. Describe the *schema* instead — column
> names, dtypes, what a row means — which is all an assistant needs to write
> correct code.

## Setup

Run these two cells once, top to bottom, before anything else.

In [ ]:
# Colab already has pandas, numpy, matplotlib, statsmodels, scipy and sklearn.
# This notebook additionally needs: duckdb.
%pip install -q duckdb

In [ ]:
# --- Data -------------------------------------------------------------------
# The unit's datasets are distributed through Blackboard. Download the
# `ACFI20009_data` folder once, put it in your own Google Drive, and this cell
# will find it. Nothing licensed is ever stored in the public repository, and
# the data never leaves your personal Drive.
from pathlib import Path

try:                                    # running on Colab
    from google.colab import drive
    drive.mount("/content/drive")
    DATA = Path("/content/drive/MyDrive/ACFI20009/data")
except ModuleNotFoundError:             # running locally (convenor)
    DATA = Path("2627/data")

assert DATA.exists(), (
    f"Data folder not found at {DATA}.\n"
    "Download it from Blackboard and place it at MyDrive/ACFI20009/data"
)
print("Data:", DATA, "|", len(list(DATA.glob('*.parquet'))), "parquet files")

---

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
pd.options.display.float_format = "{:.4f}".format

In [ ]:
import duckdb
crsp = pd.read_parquet(DATA / "crsp_monthly.parquet")

duckdb.sql(f"""
    SELECT COUNT(*) AS n_rows, COUNT(DISTINCT permno) AS n_firms
    FROM '{DATA}/crsp_monthly.parquet'
""").df()

The first question for any new table is its **grain** — what does one row represent? About 1.93 million rows and 18,266 firms: one row is one stock-**month**. Replaying the lecture's AI failure makes the difference concrete:

In [ ]:
duckdb.sql(f"""
    SELECT COUNT(permno) AS n_rows_1997,
           COUNT(DISTINCT permno) AS n_firms_1997
    FROM '{DATA}/crsp_monthly.parquet'
    WHERE date_part('year', date) = 1997
""").df()

88,590 versus 8,042 — an 11× error from one missing keyword, with perfect syntax. Roughly 8,000 firms × 11 months average ≈ 88,000 rows, so the wrong number even *explains itself* if you do the back-of-envelope. This is the row-count check from the lecture's validation protocol; it does the real work in section 7, *Verify the AI*.

### 1. SELECT and WHERE, plus the units trap

Apple's PERMNO is 14593 (everything in this unit is keyed on PERMNOs — tickers churn, PERMNOs do not):

In [ ]:
duckdb.sql(f"""
    SELECT date, ret, prc, mktcap
    FROM '{DATA}/crsp_monthly.parquet'
    WHERE permno = 14593 AND date_part('year', date) = 2021
    ORDER BY date
    LIMIT 3
""").df()

Is Apple's market cap here \$2.2 billion or \$2.2 trillion? `mktcap` ≈ `2.2e9` and the unit is **thousands of dollars**, so \$2.2 **trillion**. Every threshold you write this term must respect this: "market cap above \$1 billion" is `mktcap > 1e6`.

### 2. GROUP BY, then validate by a second route

The same panel answers a point-in-time question — how many firms were listed in December 1997:

In [ ]:
duckdb.sql(f"""
    SELECT COUNT(DISTINCT permno) AS n_dec1997
    FROM '{DATA}/crsp_monthly.parquet'
    WHERE date = DATE '1997-12-31'
""").df()

Recomputing the same number in pandas is step 2 of the validation protocol — two independent routes to one number:

In [ ]:
n_pandas = crsp.loc[crsp["date"] == "1997-12-31", "permno"].nunique()
print(f"pandas says: {n_pandas:,}")

Both routes give **7,374**. Note it is below the 8,042 firms that appeared *at some point* during 1997 — "listed in December" and "listed at any time that year" are different questions, and SQL will faithfully answer whichever you ask. Precision about the question is your job, not the database's.

The point-in-time series settles the lecture poll:

In [ ]:
duckdb.sql(f"""
    SELECT date_part('year', date) AS year,
           COUNT(DISTINCT permno) AS n_firms
    FROM '{DATA}/crsp_monthly.parquet'
    WHERE date_part('year', date) IN (1997, 2010, 2021)
    GROUP BY year
    ORDER BY year
""").df()

The US listed universe nearly halved from its 1997 peak (the Doidge–Karolyi–Stulz "listing gap"). Keep these magnitudes in your head — they are the anchors the AI gets checked against in section 7.

---

### 3. Trillion-dollar hunting (SELECT / WHERE / ORDER BY)

Now `SELECT`, `WHERE` and `ORDER BY` go to work on the month-end **2021-12-31**. The first query lists the five largest firms by market cap, showing `permno`, `prc` and market cap **in billions of dollars**; the second counts how many firms were worth more than **\$10 billion**. Both turn on the units.

In [ ]:
duckdb.sql(f"""
    SELECT permno, prc, mktcap / 1e6 AS mktcap_bn
    FROM '{DATA}/crsp_monthly.parquet'
    WHERE date = DATE '2021-12-31'
    ORDER BY mktcap DESC
    LIMIT 5
""").df()

In [ ]:
duckdb.sql(f"""
    SELECT COUNT(*) AS n_over_10bn
    FROM '{DATA}/crsp_monthly.parquet'
    WHERE date = DATE '2021-12-31' AND mktcap > 1e7
""").df()

- In order, the largest are permno 14593 (Apple, ≈ \$2,902bn), 10107 (Microsoft, ≈ \$2,525bn), 84788 (Amazon, ≈ \$1,697bn), 93436 (Tesla, ≈ \$1,092bn) and 14542 (Alphabet class C, ≈ \$916bn). Since `mktcap` is in \$thousands, dividing by `1e6` gives billions.
- **593 firms.** \$10 billion in thousands is `1e7` — write `1e10` instead and you get 0 firms, and that absurd answer *is* the units lesson: a result of zero from a reasonable question should trigger a units check, not a shrug.
- Row 5 is permno **14542 (GOOG)**; its sibling **90319 (GOOGL, ≈ \$871bn)** sits just outside the top five, at row 6 — both are Alphabet: two *share classes*, two securities, two PERMNOs, one company. Add them and Alphabet (≈ \$1.79trn) overtakes Tesla. PERMNO identifies a **security**, not a firm — exactly the right behaviour for return calculations, and something to remember whenever you count "companies". (CRSP's PERMCO exists for the firm level; our snapshot does not carry it.)

---

### 4. Exchanges in a wild year (GROUP BY)

`GROUP BY` earns its keep on calendar year **2020**. One query reports, per exchange (`exchcd`), the number of distinct firms and the average monthly return, with a `CASE` expression printing exchange names rather than codes.

The interesting part is not which exchange had the highest average monthly return, but why that win means nothing — two distinct data-quality reasons from this week's lecture apply.

In [ ]:
duckdb.sql(f"""
    SELECT CASE exchcd WHEN 1 THEN 'NYSE'
                       WHEN 2 THEN 'AMEX'
                       WHEN 3 THEN 'Nasdaq' END AS exchange,
           COUNT(DISTINCT permno) AS n_firms,
           AVG(ret) AS avg_monthly_ret
    FROM '{DATA}/crsp_monthly.parquet'
    WHERE date_part('year', date) = 2020
    GROUP BY exchcd
    ORDER BY exchcd
""").df()

Roughly: NYSE 1,308 firms, average monthly return ≈ 2.4%; AMEX 162 firms, ≈ 5.2%; Nasdaq 2,414 firms, ≈ 3.9%.

AMEX "wins", but do not buy the conclusion:

1. **Equal weighting of stock-months.** `AVG(ret)` weights a micro-cap the same as Apple. AMEX hosts tiny speculative firms whose huge percentage rebounds after March 2020 dominate an unweighted mean; in dollar (value-weighted) terms the picture differs entirely.
2. **The graveyard.** 2020's delistings — concentrated among exactly these small firms — often lack a final monthly return, so the worst months are missing from `AVG(ret)`. The average is computed over the months that survived to be recorded (the delisting bias from the lecture, live in your own query).

A good habit: whenever you report a group mean, state the weighting and what happened to exits.

---

### 5. Industry structure over two decades (JOIN)

`crsp_monthly` carries a SIC code but no industry name, so the lookup below is built in pandas and joined on a *range*. One DuckDB query then counts distinct firms per division at **2000-12-31** and at **2021-12-31** — `COUNT(DISTINCT CASE WHEN ... THEN permno END)` gets both dates in a single pass.

In [ ]:
sic_lookup = pd.DataFrame({
    "sic_lo": [100, 1000, 1500, 2000, 4000, 5000, 5200, 6000, 7000, 9100],
    "sic_hi": [999, 1499, 1799, 3999, 4999, 5199, 5999, 6799, 8999, 9999],
    "division": ["Agriculture", "Mining", "Construction", "Manufacturing",
                 "Transport & Utilities", "Wholesale", "Retail",
                 "Finance & Real Estate", "Services", "Public Administration"],
})

The join, and the two counts:

In [ ]:
duckdb.sql(f"""
    SELECT l.division,
           COUNT(DISTINCT CASE WHEN c.date = DATE '2000-12-31'
                               THEN c.permno END) AS n_2000,
           COUNT(DISTINCT CASE WHEN c.date = DATE '2021-12-31'
                               THEN c.permno END) AS n_2021
    FROM '{DATA}/crsp_monthly.parquet' c
    JOIN sic_lookup l ON c.siccd BETWEEN l.sic_lo AND l.sic_hi
    WHERE c.date IN (DATE '2000-12-31', DATE '2021-12-31')
    GROUP BY l.division
    ORDER BY n_2000 DESC
""").df()

Manufacturing shrinks from ≈ 2,333 firms to ≈ 985, Services from ≈ 1,424 to ≈ 518, Finance & Real Estate from ≈ 1,168 to ≈ 642 — the listing gap is broad-based, not sectoral.

The audit:

In [ ]:
duckdb.sql(f"""
    SELECT
      (SELECT COUNT(*) FROM '{DATA}/crsp_monthly.parquet'
        WHERE date = DATE '2021-12-31') AS rows_before,
      (SELECT COUNT(*)
         FROM '{DATA}/crsp_monthly.parquet' c
         JOIN sic_lookup l ON c.siccd BETWEEN l.sic_lo AND l.sic_hi
        WHERE c.date = DATE '2021-12-31') AS rows_after
""").df()

- **4,325 rows before, 4,304 after — the inner join silently dropped 21 firms** whose SIC code falls below 100 (missing/zero codes match no interval). Joins do not only multiply rows; they also swallow them. Check the count after *every* join, in both directions.
- **"Public Administration" shows ≈ 1,533 firms in 2021** (versus zero in 2000) — the US stock market has not been nationalised. `siccd = 9999` is CRSP's code for *unclassified*, and it lands inside the 9100–9999 interval. The join executed perfectly and the label lies; in real work you would map 9900–9999 to "Unclassified" in the lookup. Data quality is not only about values — reference tables can lie too, and you will reuse this exact join, so carry the fix with you.

---

### 6. Winsorisation done properly

`winsorise(s, p=0.01)` clips a Series at the `p` and `1 − p` quantiles — one line of pandas, with the whole judgement hiding in *where* the quantiles are computed. The first pass takes the April 2020 cross-section and winsorises it per date:

In [ ]:
def winsorise(s, p=0.01):
    lo, hi = s.quantile([p, 1 - p])
    return s.clip(lo, hi)

apr = crsp.loc[crsp["date"] == "2020-04-30", "ret"].dropna()
before_after = pd.DataFrame({"raw": apr.describe(), "per-date": winsorise(apr).describe()})
before_after.loc[["count", "mean", "std", "min", "max"]]

≈ 3,569 observations; the mean barely moves (≈ 0.193 → 0.184) while the std collapses from ≈ 0.33 to ≈ 0.24 and the max falls from ≈ **+777%** (a real micro-cap month, not a typo) to ≈ +128%. Winsorisation is gentle on location, brutal on dispersion and extremes — exactly the quantities your risk estimates depend on. Report both columns; that table *is* the disclosure.

Now the trap. Below is the same April cross-section, clipped with bounds computed from the **whole panel** (pooled) rather than from April alone, with March 2020 alongside for contrast:

In [ ]:
pool_lo, pool_hi = crsp["ret"].quantile([0.01, 0.99])
pooled = apr.clip(pool_lo, pool_hi)
mar = crsp.loc[crsp["date"] == "2020-03-31", "ret"].dropna()
print(f"Pooled bounds: [{pool_lo:.3f}, {pool_hi:.3f}]")
print(f"April 2020 pooled-clipped:  mean {pooled.mean():.4f}, std {pooled.std():.4f}")
print(f"Share of MARCH 2020 clipped from below by pooled bounds: {(mar < pool_lo).mean():.1%}")

Pooled bounds (≈ [−0.42, +0.63]) are calibrated to *average* months, so in a crisis they stop being a 1% tail-trim and become a rewrite: they clip roughly **16%** of all firms in March 2020 from below. Per-date winsorisation clips 1% of each cross-section by construction, so extreme *months* survive while extreme *observations within a month* are tamed — the outlier treatment adapts to the market, not the other way round.

---

### 7. Verify the AI (the planted flaw)

You ask an AI assistant: *"Using DuckDB on crsp_monthly.parquet, count how many US common stocks were listed at the end of 1995 and at the end of 2021."* It replies, complete with a confident explanation:

> "Here you go. We restrict to the US stock exchanges via `exchcd` and count distinct securities at each year-end:"

In [ ]:
duckdb.sql(f"""
    SELECT date, COUNT(DISTINCT permno) AS n_stocks
    FROM '{DATA}/crsp_monthly.parquet'
    WHERE date IN (DATE '1995-12-31', DATE '2021-12-31')
      AND exchcd IN (1, 2)   -- NYSE and NYSE American: the US exchanges
    GROUP BY date
    ORDER BY date
""").df()

The syntax is perfect and it avoids the lecture's `COUNT` blunder — so it gets the lecture's three-step validation protocol: a magnitude check against known anchors, a second route computed in pandas without copying the AI's filter logic, and an anchor to a documented fact.

Running it exactly as the AI wrote it:

In [ ]:
duckdb.sql(f"""
    SELECT date, COUNT(DISTINCT permno) AS n_stocks
    FROM '{DATA}/crsp_monthly.parquet'
    WHERE date IN (DATE '1995-12-31', DATE '2021-12-31')
      AND exchcd IN (1, 2)   -- the AI's filter, reproduced faithfully
    GROUP BY date
    ORDER BY date
""").df()

The query returns **2,258 firms in 1995 and 1,543 in 2021**. Both anchors scream: the demo showed **7,374** firms in December 1997 and **4,507** at some point in 2021. The AI's numbers are barely a third of the expected magnitude — fail.

Second route, in pandas, from the snapshot itself (which is already filtered to the standard universe, so no exchange filter is needed at all):

In [ ]:
n_2021 = crsp.loc[crsp["date"] == "2021-12-31", "permno"].nunique()
print(f"pandas, December 2021: {n_2021:,} firms")

About **4,325 firms** — almost three times the AI's 2021 figure.

The flaw: `exchcd IN (1, 2)` keeps NYSE and AMEX and **silently drops Nasdaq (`exchcd = 3`)** — the exchange hosting the *majority* of listed US common stocks. The comment is confidently wrong: code 2 is AMEX (NYSE American), and Nasdaq is no less a "US exchange" than either. Worse, our snapshot is already filtered to `exchcd ∈ {1, 2, 3}`, so the correct query needs **no exchange filter at all** — the AI added a restriction nobody asked for and then mislabelled it:

In [ ]:
duckdb.sql(f"""
    SELECT date, COUNT(DISTINCT permno) AS n_stocks
    FROM '{DATA}/crsp_monthly.parquet'
    WHERE date IN (DATE '1995-12-31', DATE '2021-12-31')
    GROUP BY date
    ORDER BY date
""").df()

Because Nasdaq's share of listings was even larger in the 1990s than today, the flawed query also *understates the 1995 count proportionally more* — it would shrink your measured listing gap. And note what caught it: not reading the SQL (the syntax is immaculate, the comment plausible) but the **magnitude check against known anchors**. Write this episode up as evidence → diagnosis → fix — that three-line structure is how a caught error gets reported.

---

### 8. The graveyard, quantified

Two numbers finish the report card: the fraction of firms ever in the panel that are no longer listed at the snapshot's end (2022-03), and what the leavers earned on the way out. The cell below computes the first and takes the second from `crsp_delist.parquet` where that file is present, falling back on the published magnitudes where it is not.

In [ ]:
ever = crsp["permno"].nunique()
alive = crsp.loc[crsp["date"] == crsp["date"].max(), "permno"].nunique()
print(f"Ever listed: {ever:,}   Alive 2022-03: {alive:,}   Gone: {1 - alive / ever:.0%}")

delist_path = DATA / "crsp_delist.parquet"
if delist_path.exists():
    dl = pd.read_parquet(delist_path)
    perf = dl[dl["dlstcd"].between(400, 599)]
    print(f"Performance-related delistings: {len(perf):,}, "
          f"mean delisting return: {perf['dlret'].mean():.1%}")
else:
    print("Delisting snapshot not present — published magnitudes:")
    print("  Shumway (1997): about -30% mean (NYSE/AMEX)")
    print("  Shumway & Warther (1999): about -55% (Nasdaq)")

**18,266 firms ever; 4,411 alive in March 2022; about 76% gone.**

Written out for a reader: *"Roughly three-quarters of the US common stocks that traded since 1990 have since delisted, and firms delisted for poor performance lost on average a further 30–55% in their final, often unrecorded, month. A universe built from today's listed firms therefore excludes most historical failures and omits their worst returns, so any strategy backtested on it inherits an upward bias no investor could have captured. The bias is largest for equal-weighted and small-cap portfolios, where failing firms carry the greatest weight and delist most often."*

## Before you leave
#### ⚠️ Common mistakes this week
1. **`COUNT(permno)` when you mean `COUNT(DISTINCT permno)`** — counting stock-months as stocks. Grain first, query second.
2. **Units:** `shrout` and `mktcap` are in **thousands**. "\$1bn" is `1e6`, "\$1trn" is `1e9`. A zero-row result from a sensible question usually means a units error.
3. **Pooled winsorisation.** Bounds computed on the whole panel rewrite crisis months (≈ 16% of March 2020 clipped). Winsorise within each date, and always show before/after.
4. **Unaudited joins.** Inner joins silently drop non-matching rows (21 firms here) and bad reference tables silently mislabel (1,500 "Public Administration" firms that are really unclassified). Count rows before and after; eyeball the categories.
5. **Trusting fluent SQL.** The AI's query in section 7 was syntactically perfect, confidently commented, and off by 3×. Magnitude checks against known anchors — not code-reading — caught it. Run the three-step protocol every time.